# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://mlcommons.org/croissant/) library and its Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# If `mlcroissant` is not installed, install it:
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata as well as records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (object, not dictionary)
print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Dataset description: {dataset.metadata.description}\n")

## 2. Data Overview
List the available record sets in the dataset and their field `@id`s. All elements are referenced by their `@id` according to the Croissant schema.

In [ ]:
# List available record sets and their fields
record_set_ids = []
print("Available Record Sets:")
for rs in dataset.record_sets:
    print(f"- Record set @id: {rs.id}, name: {getattr(rs, 'name', '(no name)')}")
    record_set_ids.append(rs.id)
    if rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - Field @id: {f.id}, name: {getattr(f, 'name', '(no name)')}, type: {getattr(f, 'data_type', '(unknown type)')}")
    else:
        print("  (No fields listed)")
    print("")

# Print an example record for each record set (using only the first few for demonstration):
for rs_id in record_set_ids[:2]:
    print(f"Example for record set {rs_id}:")
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            print(recs[0])
        else:
            print("  (No records)")
    except Exception as e:
        print(f"  Could not load records: {e}")
    print("")

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. Fields and columns are referenced by their `@id`. This enables programmatic selection and processing of data elements.

In [ ]:
# Extract all data into DataFrames, using @id for both record sets and fields
dataframes = {}

for rs in dataset.record_sets:
    recs = list(dataset.records(record_set=rs.id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs.id] = df
        print(f"Loaded record set: {rs.id} with shape {df.shape}")
        print(f"Columns (@id): {list(df.columns)}\n")

# Pick a specific record set (use first as an example for demonstration)
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    df_example = dataframes[example_record_set_id]
    print(f"Columns in example record set ({example_record_set_id}):")
    print(df_example.columns.tolist())
    display(df_example.head())

## 4. Exploratory Data Analysis (EDA)
We will demonstrate data selection, normalization, and grouping using column and field `@id`s. Operations such as filtering, normalization, and grouping are shown using programmatic selection via Croissant schema identifiers.

In [ ]:
# Example EDA on numeric and categorical fields using their @id
# Adjust these to actual @id values from overview (modify as per data):

target_rs_id = example_record_set_id
df = dataframes[target_rs_id]

# Try to auto-detect numeric fields (int/float), fallback to user selection
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields (@id) detected: {numeric_candidates}")

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Pick the first numeric field
else:
    # Fallback or skip EDA if no numeric field
    print("No numeric fields available for EDA.")
    numeric_field_id = None

if numeric_field_id is not None:
    # Example filter (using quantile to pick sensible threshold):
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile): {len(filtered_df)} records")

    # Normalize selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nFirst 5 rows with normalized values:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to detect a categorical/group field (object type)
    group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"\nGroupable fields (@id): {group_candidates}")
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped statistics by {group_field_id}:")
        print(grouped[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

## 5. Visualization
Visualize data distributions or relationships using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distributions if previous EDA succeeded
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field exists, plot groupwise mean of numeric field
    if group_candidates:
        group_stats = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        plt.figure(figsize=(10,4))
        group_stats.plot(kind='bar')
        plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
- Using the Croissant schema and `mlcroissant`, we loaded dataset metadata and records, referenced all data elements by their `@id`, and performed basic exploration/visualization.
- This workflow ensures transparent, auditable data processing.
- For further analysis, reference fields by their `@id` as identified in Section 2, and consult the dataset's documentation for field semantics.